# Tray Grid Classifier (cols×rows) + Grid Overlay + Cell-ID Mask

This notebook trains a **tray grid type classifier** using the labeling tool outputs and generates **grid overlays**, **cell crops**, and a **cell-id mask** on rectified tray images.

## Assumptions (matches your repo)
- Labels: `data/labels/*.labels.json`
- Rectified images: `data/rectified/*rectified*.jpg` (matched by tray id)
- Each type tuple is **(num_cols, num_rows)** i.e. `(C, R)`.
- Label JSON contains at least: `{"rows": <int>, "cols": <int>, "warp_size": {"w":..., "h":...}}`.

## Outputs
- Trained model checkpoint: `models/best_traytype_net.pth`
- Debug overlays / masks saved to `outputs/`




In [ ]:
# If needed, install deps
# !pip -q install timm

import os, json, random
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any

import numpy as np
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from tqdm.auto import tqdm


In [ ]:
# -----------------
# Config
# -----------------
@dataclass
class CFG:
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Model input size (letterbox keeps aspect ratio)
    in_h: int = 384
    in_w: int = 640

    batch_size: int = 16
    num_workers: int = 0
    epochs: int = 30

    lr_head: float = 3e-3
    lr_backbone: float = 3e-4
    weight_decay: float = 1e-4
    label_smoothing: float = 0.05

    # Optional grid alignment refinement
    do_refine: bool = True
    refine_dx: int = 12
    refine_dy: int = 12
    refine_step: int = 2
    refine_scales: Tuple[float, ...] = (0.98, 0.99, 1.0, 1.01, 1.02)

    # Unknown / reject thresholds (tune)
    min_prob: float = 0.60
    min_align_score: float = 0.08

cfg = CFG()

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
print('Device:', cfg.device)


In [ ]:
# -----------------
# Paths (repo-relative)
# -----------------
REPO_ROOT = Path('..').resolve()
DATA_DIR = REPO_ROOT / 'data'
LABELS_DIR = DATA_DIR / 'labels'
RECT_DIR = DATA_DIR / 'rectified'
OUT_DIR = REPO_ROOT / 'outputs'
MODEL_DIR = REPO_ROOT / 'models'

OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('LABELS_DIR:', LABELS_DIR)
print('RECT_DIR:', RECT_DIR)
print('OUT_DIR:', OUT_DIR)
print('MODEL_DIR:', MODEL_DIR)


In [ ]:
# -----------------
# Image preprocessing: letterbox + ImageNet normalization
# -----------------
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def resize_letterbox(img_rgb: np.ndarray, out_h: int, out_w: int, fill=0):
    """Resize with aspect ratio preserved + padding to (out_h, out_w)."""
    h, w = img_rgb.shape[:2]
    scale = min(out_w / w, out_h / h)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    resized = cv2.resize(img_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)

    canvas = np.full((out_h, out_w, 3), fill, dtype=np.uint8)
    pad_left = (out_w - new_w) // 2
    pad_top = (out_h - new_h) // 2
    canvas[pad_top:pad_top+new_h, pad_left:pad_left+new_w] = resized
    meta = {'scale': scale, 'pad_left': pad_left, 'pad_top': pad_top, 'new_w': new_w, 'new_h': new_h}
    return canvas, meta

def to_tensor_norm(img_rgb_uint8: np.ndarray) -> torch.Tensor:
    x = img_rgb_uint8.astype(np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    x = np.transpose(x, (2, 0, 1))  # CHW
    return torch.tensor(x, dtype=torch.float32)


In [ ]:
# -----------------
# Label space: class = (cols, rows, warp_w, warp_h)
# (Because your dataset contains multiple warp sizes / aspects.)
# -----------------
def type_key_from_label(j: Dict[str, Any]) -> Tuple[int, int, int, int]:
    cols = int(j['cols'])
    rows = int(j['rows'])
    ww = int(j.get('warp_size', {}).get('w', 0))
    wh = int(j.get('warp_size', {}).get('h', 0))
    return (cols, rows, ww, wh)

def pretty_key(k: Tuple[int,int,int,int]) -> str:
    c,r,ww,wh = k
    return f'{c}x{r} @ {ww}x{wh}'

# Optional: enforce nursery inventory (closed set). Keep tuple as (cols, rows).
ALLOWED_TYPES = sorted({
    (6,2), (6,6), (6,7), (6,8), (6,12), (6,17),
    (7,3), (7,5), (7,6), (7,7), (7,8), (7,9), (7,10), (7,11), (7,13), (7,15), (7,17),
    (8,16), (10,20), (12,24), (12,6),
})
ALLOWED_COLS = sorted({c for c, _ in ALLOWED_TYPES})
ALLOWED_ROWS = sorted({r for _, r in ALLOWED_TYPES})

print('Allowed cols:', ALLOWED_COLS)
print('Allowed rows:', ALLOWED_ROWS)
print('Allowed types:', len(ALLOWED_TYPES))


In [ ]:
# -----------------
# Dataset discovery: match labels to rectified images
# -----------------
def tray_id_from_label_path(label_path: Path) -> str:
    # e.g. tray_0003.jpg.labels.json -> tray_0003.jpg
    name = label_path.name
    if name.endswith('.labels.json'):
        return name[:-len('.labels.json')]
    return label_path.stem

def find_rectified_for_tray_id(tray_id: str) -> Path | None:
    # Try common patterns
    candidates = [
        RECT_DIR / f'{tray_id}.rectified.jpg',
        RECT_DIR / f'{tray_id}.rectified.png',
        RECT_DIR / f'{tray_id}.jpg.rectified.jpg',
        RECT_DIR / f'{tray_id}.jpg.rectified.png',
        RECT_DIR / f'{tray_id}.rectified.jpeg',
    ]
    for p in candidates:
        if p.exists():
            return p

    # Fallback: search by prefix
    hits = sorted(RECT_DIR.glob(f'{tray_id}*rectified*'))
    return hits[0] if hits else None

def build_items(labels_dir: Path) -> List[Dict[str, Path]]:
    items = []
    for lp in sorted(labels_dir.glob('*.labels.json')):
        tray_id = tray_id_from_label_path(lp)
        rp = find_rectified_for_tray_id(tray_id)
        if rp is None:
            continue
        items.append({'label_path': lp, 'img_path': rp})
    return items

items = build_items(LABELS_DIR)
print('Found labeled pairs:', len(items))
if items[:3]:
    print('Example:', items[0])


In [ ]:
# -----------------
# Build class vocab from labeled data (+ optional nursery filter)
# -----------------
def load_json(p: Path) -> Dict[str, Any]:
    return json.loads(p.read_text())

keys = []
for it in items:
    j = load_json(it['label_path'])
    # Enforce nursery inventory only if desired:
    # if (j['cols'], j['rows']) not in ALLOWED_TYPES: continue
    keys.append(type_key_from_label(j))

uniq_keys = sorted(set(keys))
key_to_idx = {k:i for i,k in enumerate(uniq_keys)}
idx_to_key = uniq_keys

print('Num classes:', len(idx_to_key))
for k in idx_to_key[:10]:
    print(' -', pretty_key(k))


In [ ]:
# -----------------
# Split train/val by class (stratified-ish)
# -----------------
from collections import defaultdict

def split_items(items: List[Dict[str, Path]], val_frac: float = 0.2):
    by_k = defaultdict(list)
    for it in items:
        j = load_json(it['label_path'])
        k = type_key_from_label(j)
        # if (j['cols'], j['rows']) not in ALLOWED_TYPES: continue
        by_k[k].append(it)

    train_items, val_items = [], []
    for k, group in by_k.items():
        random.shuffle(group)
        n = len(group)
        if n <= 1:
            train_items += group
        else:
            split = max(1, int(round(val_frac * n)))
            val_items += group[:split]
            train_items += group[split:]

    return train_items, val_items

train_items, val_items = split_items(items, val_frac=0.2)
print('Train:', len(train_items), 'Val:', len(val_items))


In [ ]:
# -----------------
# Dataset
# -----------------
class TrayTypeDataset(Dataset):
    def __init__(self, items: List[Dict[str, Path]], key_to_idx: Dict[Tuple[int,int,int,int], int], train: bool=True):
        self.items = items
        self.key_to_idx = key_to_idx
        self.train = train

    def __len__(self):
        return len(self.items)

    def _augment(self, img_rgb: np.ndarray) -> np.ndarray:
        # Light, safe augs for structure recognition
        if random.random() < 0.5:
            img_rgb = img_rgb[:, ::-1]

        if random.random() < 0.7:
            alpha = 1.0 + random.uniform(-0.2, 0.2)
            beta  = random.uniform(-20, 20)
            img_rgb = np.clip(alpha * img_rgb + beta, 0, 255).astype(np.uint8)

        if random.random() < 0.2:
            k = random.choice([3,5])
            img_rgb = cv2.GaussianBlur(img_rgb, (k,k), 0)

        if random.random() < 0.3:
            # Random erase to prevent shortcut learning from plant canopy
            H,W = img_rgb.shape[:2]
            er_w = int(W * random.uniform(0.05, 0.15))
            er_h = int(H * random.uniform(0.05, 0.15))
            x0 = random.randint(0, max(0, W-er_w))
            y0 = random.randint(0, max(0, H-er_h))
            img_rgb[y0:y0+er_h, x0:x0+er_w] = 0

        return img_rgb

    def __getitem__(self, idx):
        it = self.items[idx]
        img_bgr = cv2.imread(str(it['img_path']))
        if img_bgr is None:
            raise FileNotFoundError(it['img_path'])
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        j = load_json(it['label_path'])
        k = type_key_from_label(j)
        y = self.key_to_idx[k]

        img_rgb, _ = resize_letterbox(img_rgb, cfg.in_h, cfg.in_w, fill=0)
        if self.train:
            img_rgb = self._augment(img_rgb)

        x = to_tensor_norm(img_rgb)
        return x, torch.tensor(y, dtype=torch.long)

train_ds = TrayTypeDataset(train_items, key_to_idx, train=True)
val_ds   = TrayTypeDataset(val_items, key_to_idx, train=False)

train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
val_dl   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

len(train_ds), len(val_ds)


In [ ]:
# -----------------
# Model
# -----------------
class TrayTypeNet(nn.Module):
    def __init__(self, backbone_name: str, n_classes: int):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0, global_pool='avg')
        feat = self.backbone.num_features
        self.head = nn.Linear(feat, n_classes)

    def forward(self, x):
        f = self.backbone(x)
        return self.head(f)

def smooth_ce(logits, targets, smoothing=0.0):
    if smoothing <= 0:
        return F.cross_entropy(logits, targets)
    n = logits.size(1)
    logp = F.log_softmax(logits, dim=1)
    with torch.no_grad():
        true_dist = torch.zeros_like(logp)
        true_dist.fill_(smoothing / (n - 1))
        true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing)
    return torch.mean(torch.sum(-true_dist * logp, dim=1))

model = TrayTypeNet('resnet18', n_classes=len(idx_to_key)).to(cfg.device)

optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': cfg.lr_backbone},
    {'params': model.head.parameters(), 'lr': cfg.lr_head},
], weight_decay=cfg.weight_decay)

print('Model params:', sum(p.numel() for p in model.parameters())/1e6, 'M')


In [ ]:
# -----------------
# Train / Evaluate
# -----------------
@torch.no_grad()
def eval_acc(model, dl):
    model.eval()
    total, correct = 0, 0
    for x,y in dl:
        x = x.to(cfg.device)
        y = y.to(cfg.device)
        logits = model(x)
        pred = logits.argmax(1)
        total += x.size(0)
        correct += (pred == y).sum().item()
    return correct / total if total else 0.0

best = 0.0
ckpt_path = MODEL_DIR / 'best_traytype_net.pth'

for epoch in range(1, cfg.epochs+1):
    model.train()
    pbar = tqdm(train_dl, desc=f'Epoch {epoch}/{cfg.epochs}')
    running = 0.0

    for x,y in pbar:
        x = x.to(cfg.device)
        y = y.to(cfg.device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = smooth_ce(logits, y, cfg.label_smoothing)
        loss.backward()
        optimizer.step()

        running += loss.item()
        pbar.set_postfix(loss=running / (pbar.n + 1))

    acc = eval_acc(model, val_dl)
    print('Val acc:', acc)

    if acc > best:
        best = acc
        torch.save({
            'model': model.state_dict(),
            'idx_to_key': [list(k) for k in idx_to_key],
            'backbone': 'resnet18',
            'cfg': cfg.__dict__,
        }, ckpt_path)
        print('Saved best:', best, '->', ckpt_path)

print('Best val acc:', best)


In [ ]:
# -----------------
# Inference helpers
# -----------------
def load_checkpoint(path: Path):
    ckpt = torch.load(path, map_location=cfg.device)
    idx_to_key = [tuple(x) for x in ckpt['idx_to_key']]
    backbone = ckpt.get('backbone', 'resnet18')
    m = TrayTypeNet(backbone, n_classes=len(idx_to_key)).to(cfg.device)
    m.load_state_dict(ckpt['model'])
    m.eval()
    return m, idx_to_key

@torch.no_grad()
def predict_type(m, idx_to_key, rectified_bgr: np.ndarray):
    rgb = cv2.cvtColor(rectified_bgr, cv2.COLOR_BGR2RGB)
    rgb, _ = resize_letterbox(rgb, cfg.in_h, cfg.in_w, fill=0)
    x = to_tensor_norm(rgb).unsqueeze(0).to(cfg.device)
    logits = m(x)
    probs = F.softmax(logits, dim=1)[0].detach().cpu().numpy()
    idx = int(probs.argmax())
    return idx_to_key[idx], float(probs[idx]), probs

m_infer, idx_to_key_infer = load_checkpoint(ckpt_path)
print('Loaded classes:', len(idx_to_key_infer))


In [ ]:
# -----------------
# Grid rendering outputs: overlay + cell-id mask + crops
# -----------------
def grid_boxes(W, H, cols, rows, dx=0, dy=0, scale=1.0):
    cx, cy = W/2, H/2
    cell_w = (W / cols) * scale
    cell_h = (H / rows) * scale
    grid_w = cell_w * cols
    grid_h = cell_h * rows
    x_start = cx - grid_w/2 + dx
    y_start = cy - grid_h/2 + dy

    boxes = []
    for r in range(rows):
        for c in range(cols):
            x0 = int(round(x_start + c*cell_w))
            x1 = int(round(x_start + (c+1)*cell_w))
            y0 = int(round(y_start + r*cell_h))
            y1 = int(round(y_start + (r+1)*cell_h))
            boxes.append((x0,y0,x1,y1,r,c))
    return boxes

def draw_grid_overlay(img_bgr, cols, rows, dx=0, dy=0, scale=1.0, color=(255,255,0), thickness=2):
    H,W = img_bgr.shape[:2]
    out = img_bgr.copy()
    boxes = grid_boxes(W,H,cols,rows,dx=dx,dy=dy,scale=scale)
    xs = sorted(set([b[0] for b in boxes] + [b[2] for b in boxes]))
    ys = sorted(set([b[1] for b in boxes] + [b[3] for b in boxes]))
    for x in xs:
        cv2.line(out, (x,0), (x,H-1), color, thickness)
    for y in ys:
        cv2.line(out, (0,y), (W-1,y), color, thickness)
    return out

def make_cell_id_mask(img_bgr, cols, rows, dx=0, dy=0, scale=1.0):
    H,W = img_bgr.shape[:2]
    mask = np.zeros((H,W), dtype=np.int32)
    for (x0,y0,x1,y1,r,c) in grid_boxes(W,H,cols,rows,dx=dx,dy=dy,scale=scale):
        xx0,yy0 = max(0,x0), max(0,y0)
        xx1,yy1 = min(W,x1), min(H,y1)
        if xx1<=xx0 or yy1<=yy0:
            continue
        cell_id = r*cols + c + 1
        mask[yy0:yy1, xx0:xx1] = cell_id
    return mask

def crop_cells(img_bgr, cols, rows, dx=0, dy=0, scale=1.0, shrink=0.05):
    H,W = img_bgr.shape[:2]
    crops = []
    for (x0,y0,x1,y1,r,c) in grid_boxes(W,H,cols,rows,dx=dx,dy=dy,scale=scale):
        if shrink > 0:
            sx = int((x1-x0)*shrink)
            sy = int((y1-y0)*shrink)
            x0,x1 = x0+sx, x1-sx
            y0,y1 = y0+sy, y1-sy
        xx0,yy0 = max(0,x0), max(0,y0)
        xx1,yy1 = min(W,x1), min(H,y1)
        if xx1<=xx0 or yy1<=yy0:
            continue
        crops.append({'r': r, 'c': c, 'bbox': (xx0,yy0,xx1,yy1), 'img': img_bgr[yy0:yy1, xx0:xx1].copy()})
    return crops


In [ ]:
# -----------------
# Optional refinement: align grid lines to edges
# -----------------
def edge_map(img_bgr):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (5,5), 0)
    e = cv2.Canny(g, 50, 150)
    return e.astype(np.float32) / 255.0

def line_mask(W, H, cols, rows, dx=0, dy=0, scale=1.0, thickness=2):
    m = np.zeros((H,W), dtype=np.uint8)
    boxes = grid_boxes(W,H,cols,rows,dx=dx,dy=dy,scale=scale)
    xs = sorted(set([b[0] for b in boxes] + [b[2] for b in boxes]))
    ys = sorted(set([b[1] for b in boxes] + [b[3] for b in boxes]))
    for x in xs:
        cv2.line(m, (x,0), (x,H-1), 255, thickness)
    for y in ys:
        cv2.line(m, (0,y), (W-1,y), 255, thickness)
    return (m.astype(np.float32) / 255.0)

def refine_grid_placement(img_bgr, cols, rows):
    H,W = img_bgr.shape[:2]
    E = edge_map(img_bgr)
    best = (-1e9, 0, 0, 1.0)
    for s in cfg.refine_scales:
        for dy in range(-cfg.refine_dy, cfg.refine_dy+1, cfg.refine_step):
            for dx in range(-cfg.refine_dx, cfg.refine_dx+1, cfg.refine_step):
                M = line_mask(W,H,cols,rows,dx=dx,dy=dy,scale=s,thickness=2)
                denom = M.sum() + 1e-6
                score = float((E*M).sum() / denom)
                if score > best[0]:
                    best = (score, dx, dy, s)
    score, dx, dy, s = best
    return dx, dy, s, score


In [ ]:
# -----------------
# Run inference on a single rectified image and write outputs
# -----------------
# Choose any rectified image you want (example uses first val item if available)
example_path = None
if val_items:
    example_path = val_items[0]['img_path']
elif train_items:
    example_path = train_items[0]['img_path']
else:
    raise RuntimeError('No labeled items found. Check data/labels and data/rectified.')

img_bgr = cv2.imread(str(example_path))
assert img_bgr is not None, example_path

(k, prob, _) = predict_type(m_infer, idx_to_key_infer, img_bgr)
cols, rows, warp_w, warp_h = map(int, k)
print('Pred:', pretty_key(k), 'prob:', prob)

dx=dy=0
s=1.0
align_score = None
if cfg.do_refine:
    dx, dy, s, align_score = refine_grid_placement(img_bgr, cols, rows)
    print('Refined:', {'dx': dx, 'dy': dy, 'scale': s, 'align_score': align_score})

is_unknown = (prob < cfg.min_prob) or (align_score is not None and align_score < cfg.min_align_score)
print('UNKNOWN?', is_unknown)

# Outputs
overlay = draw_grid_overlay(img_bgr, cols, rows, dx=dx, dy=dy, scale=s, color=(255,255,0), thickness=2)
mask = make_cell_id_mask(img_bgr, cols, rows, dx=dx, dy=dy, scale=s)
mask_bin = (mask > 0).astype(np.uint8) * 255

overlay_path = OUT_DIR / f'{Path(example_path).stem}.grid_debug.jpg'
mask_path = OUT_DIR / f'{Path(example_path).stem}.cell_mask_binary.png'
cv2.imwrite(str(overlay_path), overlay)
cv2.imwrite(str(mask_path), mask_bin)

cells = crop_cells(img_bgr, cols, rows, dx=dx, dy=dy, scale=s, shrink=0.05)
print('Num cells:', len(cells))

# Save a few crops
for i in [0, len(cells)//2, len(cells)-1]:
    if 0 <= i < len(cells):
        c = cells[i]
        crop_path = OUT_DIR / f'{Path(example_path).stem}.cell_r{c['r']}_c{c['c']}.jpg'
        cv2.imwrite(str(crop_path), c['img'])

print('Wrote:', overlay_path)
print('Wrote:', mask_path)
print('Crops in:', OUT_DIR)


In [ ]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate_on_loader(model, dl, device):
    model.eval()
    total, correct = 0, 0
    for x, y in tqdm(dl, desc="Eval"):
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        pred = logits.argmax(1)
        total += x.size(0)
        correct += (pred == y).sum().item()
    return correct / total if total else 0.0

val_acc = evaluate_on_loader(model, val_dl, cfg.device)
print("Val accuracy:", val_acc)

In [ ]:
@torch.no_grad()
def show_batch_predictions(model, dl, idx_to_key, device, n=10):
    model.eval()
    x, y = next(iter(dl))
    x = x.to(device)
    logits = model(x)
    probs = F.softmax(logits, dim=1)
    top = probs.argmax(1).cpu().numpy()
    y = y.numpy()
    for i in range(min(n, len(y))):
        gt = idx_to_key[int(y[i])]
        pr = idx_to_key[int(top[i])]
        print(f"{i}: GT={gt}  PRED={pr}")

show_batch_predictions(model, val_dl, idx_to_key, cfg.device, n=12)

In [ ]:
from pathlib import Path
import cv2

# pick any rectified image
img_path = Path("../data/rectified/tray_1022.jpg.rectified.jpg")  # change
img_bgr = cv2.imread(str(img_path))
assert img_bgr is not None, img_path

k, prob, _ = predict_type(m_infer, idx_to_key_infer, img_bgr)
cols, rows, warp_w, warp_h = map(int, k)
print("Pred:", (cols, rows, warp_w, warp_h), "prob:", prob)

dx=dy=0; s=1.0; align_score=None
if cfg.do_refine:
    dx, dy, s, align_score = refine_grid_placement(img_bgr, cols, rows)
    print("Refined:", dx, dy, s, "align_score:", align_score)

overlay = draw_grid_overlay(img_bgr, cols, rows, dx=dx, dy=dy, scale=s, color=(255,255,0), thickness=2)
mask = make_cell_id_mask(img_bgr, cols, rows, dx=dx, dy=dy, scale=s)

out_overlay = Path("../outputs/predictions") / f"{img_path.stem}.overlay.jpg"
# out_mask = Path("outputs") / f"{img_path.stem}.pred.mask.png"
cv2.imwrite(str(out_overlay), overlay)
# cv2.imwrite(str(out_mask), ((mask > 0).astype("uint8") * 255))

print("Wrote:", out_overlay)
# print("Wrote:", out_mask)

In [ ]:
from pathlib import Path
import cv2
from tqdm.auto import tqdm

rect_dir = Path("../data/rectified")
out_dir = Path("../outputs/predictions")
out_dir.mkdir(parents=True, exist_ok=True)

# choose subset
paths = sorted(rect_dir.glob("*rectified*.jpg"))[:100]

for p in tqdm(paths, desc="Predicting"):
    img = cv2.imread(str(p))
    if img is None:
        continue

    k, prob, _ = predict_type(m_infer, idx_to_key_infer, img)
    cols, rows, warp_w, warp_h = map(int, k)

    dx=dy=0; s=1.0
    if cfg.do_refine:
        dx, dy, s, align_score = refine_grid_placement(img, cols, rows)
    else:
        align_score = None

    overlay = draw_grid_overlay(img, cols, rows, dx=dx, dy=dy, scale=s, color=(255,255,0), thickness=2)
    cv2.imwrite(str(out_dir / f"{p.stem}.overlay.jpg"), overlay)

print("Done. Overlays in:", out_dir)

In [ ]:
from ultralytics import YOLO

# Windows notebook tip: avoid multiprocessing DataLoader crashes
# (training is already done above; this just prevents surprises if you create loaders later)
cfg.num_workers = 0


In [ ]:
# -----------------
# YOLO model (set your weights path)
# -----------------
YOLO_WEIGHTS = '../models/trayseg_v9.pt'  

yolo = YOLO(YOLO_WEIGHTS)

In [ ]:
# -----------------
# Upstream rectification utility
# -----------------
from src.rectification import predict_and_crop


In [ ]:
# -----------------
# Batch test on RAW images (no manual rectification)
# -----------------
# RAW_DIR = Path('../data/all_images')  # <-- CHANGE if your raw images live elsewhere
RAW_DIR = Path('../data/rectified')  # <-- CHANGE if your raw images live elsewhere
RAW_GLOB = '*.jpg'          # <-- CHANGE if png/jpeg

raw_paths = sorted(RAW_DIR.glob(RAW_GLOB))
print('Found raw images:', len(raw_paths))

out_rect = OUT_DIR / 'labeled_auto_rectified'
out_pred = OUT_DIR / 'labeled_auto_predictions'
out_rect.mkdir(parents=True, exist_ok=True)
out_pred.mkdir(parents=True, exist_ok=True)

# Load trained grid classifier checkpoint (from earlier cells)
m_infer, idx_to_key_infer = load_checkpoint(ckpt_path)

def decide_unknown(prob, align_score=None):
    if prob < cfg.min_prob:
        return True
    if align_score is not None and align_score < cfg.min_align_score:
        return True
    return False

start = 0
end   = 105        # inclusive number you care about
subset = raw_paths[start:end+1]   # slice handles bounds for you

# Run a small subset first
# N = min(50, len(raw_paths))
# for p in tqdm(raw_paths[:N], desc='Auto-rectify + predict'):
for p in tqdm(subset, desc='Auto-rectify + predict'):
    full = cv2.imread(str(p))
    if full is None:
        continue

    warped = predict_and_crop(full, yolo, out_w=1400, conf=0.25, iou=0.7, imgsz=1024)
    if warped is None:
        print('Rectify failed:', p.name, '-> no usable tray detection')
        continue

    # Save rectified for inspection
    rect_path = out_rect / f'{p.stem}.rectified.jpg'
    cv2.imwrite(str(rect_path), warped)

    # Predict grid type from rectified tray
    k, prob, _ = predict_type(m_infer, idx_to_key_infer, warped)
    cols, rows, warp_w, warp_h = map(int, k)

    dx=dy=0; s=1.0; align_score=None
    if cfg.do_refine:
        dx, dy, s, align_score = refine_grid_placement(warped, cols, rows)

    unk = decide_unknown(prob, align_score)

    overlay = draw_grid_overlay(warped, cols, rows, dx=dx, dy=dy, scale=s, color=(255,255,0), thickness=2)
    mask = make_cell_id_mask(warped, cols, rows, dx=dx, dy=dy, scale=s)
    mask_bin = (mask > 0).astype(np.uint8) * 255

    ov_path = out_pred / f'{p.stem}.pred_overlay_{cols}x{rows}_p{prob:.2f}.jpg'
    # mk_path = out_pred / f'{p.stem}.pred_mask_{cols}x{rows}_p{prob:.2f}.png'

    cv2.imwrite(str(ov_path), overlay)
    # cv2.imwrite(str(mk_path), mask_bin)

print('Rectified outputs:', out_rect)
print('Prediction overlays:', out_pred)


In [ ]:
# =================
# Full Classifier Evaluation
# =================
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    top_k_accuracy_score,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Load best checkpoint
m_eval, idx_to_key_eval = load_checkpoint(ckpt_path)
class_names = [pretty_key(k) for k in idx_to_key_eval]

@torch.no_grad()
def collect_preds(model, dl, device):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for x, y in tqdm(dl, desc="Evaluating"):
        x = x.to(device)
        logits = model(x)
        probs  = F.softmax(logits, dim=1).cpu().numpy()
        preds  = probs.argmax(axis=1)
        all_labels.append(y.numpy())
        all_preds.append(preds)
        all_probs.append(probs)
    return (
        np.concatenate(all_labels),
        np.concatenate(all_preds),
        np.concatenate(all_probs, axis=0),
    )

y_true, y_pred, y_probs = collect_preds(m_eval, val_dl, cfg.device)
n_classes  = len(idx_to_key_eval)
top_probs  = y_probs.max(axis=1)
correct_mask = (y_true == y_pred)

# ── 1. Accuracy ──────────────────────────────────────────────
acc = correct_mask.mean()
print(f"Accuracy          : {acc:.4f}  ({int(acc * len(y_true))}/{len(y_true)})")

# ── 2. Top-k accuracy ────────────────────────────────────────
for k in [2, 3]:
    if n_classes > k:
        topk = top_k_accuracy_score(
            y_true, y_probs, k=k,
            labels=list(range(n_classes))
        )
        print(f"Top-{k} accuracy    : {topk:.4f}")

# ── 3. Per-class report ──────────────────────────────────────
present_labels = sorted(set(y_true) | set(y_pred))
present_names  = [class_names[i] for i in present_labels]

print("\n── Per-class report ──")
report = classification_report(
    y_true, y_pred,
    labels=present_labels,
    target_names=present_names,
    zero_division=0,
)
print(report)

# ── 4. Macro / weighted averages ─────────────────────────────
for avg in ["macro", "weighted"]:
    f1 = f1_score(y_true, y_pred, average=avg, zero_division=0, labels=present_labels)
    p  = precision_score(y_true, y_pred, average=avg, zero_division=0, labels=present_labels)
    r  = recall_score(y_true, y_pred, average=avg, zero_division=0, labels=present_labels)
    print(f"{avg.capitalize():10s} — Precision: {p:.4f}  Recall: {r:.4f}  F1: {f1:.4f}")

# ── 5. Mean confidence on correct vs incorrect ───────────────
print(f"\nMean confidence (correct)  : {top_probs[correct_mask].mean():.4f}")
print(f"Mean confidence (incorrect): {top_probs[~correct_mask].mean():.4f}")

# ── 6. Confusion matrix ──────────────────────────────────────
cm = confusion_matrix(y_true, y_pred, labels=present_labels)

fig, ax = plt.subplots(figsize=(max(8, len(present_labels)), max(6, len(present_labels) * 0.75)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=present_names)
disp.plot(ax=ax, xticks_rotation=45, colorbar=True, cmap="Blues")
ax.set_title("Confusion matrix — validation set")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

# ── 7. Per-class accuracy bar chart ──────────────────────────
per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
fig, ax = plt.subplots(figsize=(10, max(4, len(present_labels) * 0.4)))
bars = ax.barh(present_names, per_class_acc, color="steelblue", edgecolor="none")
ax.bar_label(bars, fmt="%.2f", padding=4, fontsize=9)
ax.set_xlim(0, 1.15)
ax.set_xlabel("Per-class accuracy")
ax.set_title("Per-class accuracy — validation set")
ax.axvline(acc, color="tomato", linestyle="--", linewidth=1, label=f"Overall acc = {acc:.2f}")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / "per_class_accuracy.png", dpi=150)
plt.show()

# ── 8. Confidence histogram ───────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(top_probs[correct_mask],  bins=20, alpha=0.7, label="Correct",   color="steelblue")
ax.hist(top_probs[~correct_mask], bins=20, alpha=0.7, label="Incorrect", color="tomato")
ax.axvline(cfg.min_prob, color="gray", linestyle="--", linewidth=1, label=f"min_prob = {cfg.min_prob}")
ax.set_xlabel("Max softmax probability")
ax.set_ylabel("Count")
ax.set_title("Confidence distribution — correct vs incorrect")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "confidence_histogram.png", dpi=150)
plt.show()

print(f"\nAll plots saved to: {OUT_DIR}")